# 1. Two Sum
**Difficulty:** 🟢 Easy · **Topic:** Array / Hash Map · **LeetCode:** https://leetcode.com/problems/two-sum/

## Concepts

- **Core concept(s):** Hashing (hash map for O(1) lookups), and — for comparison — sorting with the **two pointers** technique.
- **Why they apply here:** For each number `x` we need to know whether its *complement* (`target - x`) has already been seen. A hash map answers "have I seen this value?" in O(1), which turns an O(n²) pairwise search into a single O(n) pass. Sorting + two pointers is a classic alternative when the array is (or can be) ordered, because a sorted array lets us shrink the search space from both ends.
- **Key intuition / mental model:** Instead of asking "for this pair, does it sum to target?" for every pair (brute force), flip the question to "for this single number, does its partner already exist?" — that reframing is what makes the hash-map solution possible.
- **Prerequisite knowledge:** basic array iteration, what a dictionary/hash table is.

### What is a Hash Map?
**What it is:** A hash map (Python `dict`) stores key → value pairs and lets you look up a value by its key without scanning the whole collection.
**How it works internally / mental picture:** It runs each key through a *hash function* that computes an index into an internal array (the "bucket"). To look up a key, it recomputes the hash and jumps straight to that bucket instead of scanning element by element. Collisions (two keys hashing to the same bucket) are resolved internally (e.g. chaining or open addressing) but don't change the average-case behavior.
**Key operations + complexity:** insert, lookup, and delete are all **O(1) on average** (O(n) worst case under many hash collisions, which is rare in practice with good hash functions).
**In Python:** use `dict` for key → value storage (e.g. value → index), or `set` when you only need membership testing (no associated value).

### What are Two Pointers?
**What it is:** A technique where two index variables scan a sequence from different positions (often one from the start, one from the end) and move inward or forward based on a comparison, instead of using nested loops.
**How it works internally / mental picture:** On a **sorted** array, if the sum of the values at `left` and `right` is too small, moving `left` rightward can only increase the sum; if it's too large, moving `right` leftward can only decrease it. This lets you discard a whole range of impossible pairs at each step rather than checking them individually.
**Key operations + complexity:** each pointer moves at most `n` times total, so a full scan is **O(n)** after the array is sorted (sorting itself costs O(n log n), which then dominates).
**In Python:** plain integer indices into a `list`, advanced with `left += 1` / `right -= 1` inside a `while left < right` loop.

## Problem Statement

Given an array of integers `nums` and an integer `target`, return the **indices** of the two numbers that add up to `target`.

- Assume exactly one valid answer exists, and you may not use the same element twice.
- The order of the returned indices does not matter (LeetCode accepts either order).

**Examples:**
```
Input:  nums = [2, 7, 11, 15], target = 9
Output: [0, 1]          # nums[0] + nums[1] == 9

Input:  nums = [3, 2, 4], target = 6
Output: [1, 2]          # nums[1] + nums[2] == 6
```

**Constraints:**
- `2 <= nums.length <= 10^4`
- `-10^9 <= nums[i], target <= 10^9`
- Exactly one valid answer exists.

### Approach 1 — Brute Force
**Idea:** Try every pair `(i, j)` with `i < j` and check whether `nums[i] + nums[j] == target`. This directly checks the definition of the problem with no extra data structures.
**Time complexity:** `O(n^2)` — two nested loops over the array, each pair examined once.
**Space complexity:** `O(1)` — no auxiliary data structure beyond the loop variables.

In [ ]:
from typing import List

def two_sum_brute(nums: List[int], target: int) -> List[int]:
    n = len(nums)
    for i in range(n):                     # fix the first number
        for j in range(i + 1, n):           # try every later number as the partner
            if nums[i] + nums[j] == target:  # direct check against the problem definition
                return [i, j]
    return []  # problem guarantees a solution exists, but keep a safe fallback

### Approach 2 — Better (Sort + Two Pointers)
**Idea:** Sort `(value, original_index)` pairs by value. Walk two pointers inward from both ends of the sorted array: if the current sum is too small, move `left` right (to increase the sum); if too large, move `right` left (to decrease it); if it matches, return the *original* indices. This avoids the pairwise nested loop but pays for sorting.
**Time complexity:** `O(n log n)` — dominated by the sort; the two-pointer scan itself is `O(n)`.
**Space complexity:** `O(n)` — for the list of `(value, original_index)` pairs created before sorting.

In [ ]:
from typing import List

def two_sum_better(nums: List[int], target: int) -> List[int]:
    # Pair each value with its original index so we can recover it after sorting.
    indexed = sorted((val, idx) for idx, val in enumerate(nums))

    left, right = 0, len(indexed) - 1
    while left < right:
        current_sum = indexed[left][0] + indexed[right][0]
        if current_sum == target:
            return sorted([indexed[left][1], indexed[right][1]])  # return original indices
        elif current_sum < target:
            left += 1   # sum too small -> need a bigger left value
        else:
            right -= 1  # sum too large -> need a smaller right value
    return []

### Approach 3 — Optimal (Hash Map)
**Idea:** Walk the array once. For each value `x`, compute its complement `target - x`. If the complement has already been recorded in a hash map (`value -> index`), we've found the pair immediately; otherwise, record `x`'s own index for future lookups. This needs only a single pass because the hash map answers "have I seen this value?" in O(1).
**Time complexity:** `O(n)` — a single pass over the array, with O(1) average-case hash map operations per element.
**Space complexity:** `O(n)` — the hash map can hold up to `n` entries in the worst case.

In [ ]:
from typing import List

def two_sum_optimal(nums: List[int], target: int) -> List[int]:
    seen = {}                       # value -> index of values we've passed
    for i, x in enumerate(nums):
        need = target - x           # the complement that would complete the pair
        if need in seen:            # O(1) membership test -- the whole point
            return [seen[need], i]
        seen[x] = i                 # record current value for future complements
    return []

## Test / Verification

In [ ]:
def normalize(pair, nums, target):
    # Order-independent check: confirm the returned indices actually sum to target.
    i, j = pair
    return i != j and nums[i] + nums[j] == target

test_cases = [
    ([2, 7, 11, 15], 9),
    ([3, 2, 4], 6),
    ([3, 3], 6),
    ([-1, -2, -3, -4, -5], -8),
    ([0, 4, 3, 0], 0),
]

solutions = {
    "brute": two_sum_brute,
    "better (sort + two pointers)": two_sum_better,
    "optimal (hash map)": two_sum_optimal,
}

for nums, target in test_cases:
    print(f"nums={nums}, target={target}")
    for label, fn in solutions.items():
        result = fn(nums, target)
        ok = normalize(result, nums, target)
        print(f"  {label:32s} -> {result}  {'OK' if ok else 'FAIL'}")
    print()

## Empirical Complexity Benchmark

Big-O can't be read directly off a single timing, but it can be **measured** via the **doubling ratio**: time each approach on inputs of growing `n` and observe how runtime scales when `n` doubles.

| Growth | Expected ratio when n doubles |
|---|---|
| O(1) / O(log n) | ~1x (barely changes) |
| O(n) / O(n log n) | ~2x (roughly linear) |
| O(n^2) | ~4x (quadratic) |
| O(n^3) | ~8x (cubic) |

We expect: brute force (`O(n^2)`) to show ~4x ratios, the sort+two-pointers approach (`O(n log n)`) to show a little over ~2x, and the hash map approach (`O(n)`) to show ~2x.

In [ ]:
import os
import sys

# Bootstrap: locate bench_utils.py by walking up from the current working
# directory, so this works regardless of where the kernel was started.
# TEST NOTE: bench_utils.py is colocated with this notebook in the same
# outputs/ folder for this evaluation run (instead of the shared
# notebooks/ root the real skill uses), so the walk-up finds it immediately.
def _find_and_add_bench_utils():
    d = os.getcwd()
    for _ in range(6):
        candidate = os.path.join(d, "bench_utils.py")
        if os.path.isfile(candidate):
            if d not in sys.path:
                sys.path.insert(0, d)
            return True
        parent = os.path.dirname(d)
        if parent == d:
            break
        d = parent
    return False

_find_and_add_bench_utils()
from bench_utils import benchmark

def make_worst_case(n):
    # No pair sums to target until the very last two elements -> forces every
    # approach to do its full amount of work (no early exit).
    nums = list(range(1, n + 1))
    target = nums[-1] + nums[-2]
    return (nums, target)

solutions_bench = {
    "brute O(n^2)": two_sum_brute,
    "better O(n log n)": two_sum_better,
    "optimal O(n)": two_sum_optimal,
}

# Keep brute force's O(n^2) cost manageable while still doubling cleanly.
sizes = [500, 1000, 2000, 4000]

benchmark(solutions_bench, make_worst_case, sizes, repeats=3, plot=True)

## 🧩 Patterns Learned

- **Hash Map for O(1) complement lookup** — whenever a problem asks "does some paired/related value already exist among the elements I've seen", trade O(n) space for a single-pass O(n) time solution instead of nested loops.
- **When to reach for this pattern:** the problem statement mentions pairs, complements, or "does X exist such that X + Y = target", especially when it forbids using the same element twice and only needs a single valid answer.
- **Sort + Two Pointers as a fallback pattern:** useful when a hash map's extra space is undesirable, when the array is already sorted, or when the problem asks for *all* pairs/triplets (as in 3Sum) where two pointers naturally avoid duplicate work after fixing one element.
- **Related problems:** 3Sum, 4Sum, Two Sum II — Input Array Is Sorted, Subarray Sum Equals K (prefix-sum + hash map variant), Contains Duplicate.
- **Common pitfalls:** forgetting that the two-pointer approach needs the array **sorted first** (and must track original indices separately); using the *same* element twice (check `i != j`); assuming a hash map lookup can't hit a worst-case O(n) collision path (rare, but why brute force is still occasionally relevant to discuss).